# Hybrid Model Confidence Intervals

This notebook computes confidence intervals for the existing 5-class hybrid SNR-aware attention model.

It does **not** use k-fold, smoothing, or monotonic trend forcing.

It reports:

- per-SNR accuracy with 95% Wilson confidence intervals
- hybrid vs normal-attention accuracy with 95% CI bands
- paired bootstrap 95% CI for the improvement: hybrid minus normal attention
- optional LSTM baseline CI using its saved per-SNR accuracy

Main result folder:

```text
experiments/5class_hybrid_confidence_intervals/
```

In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import Image, display, FileLink

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_5class.pkl'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

NORMAL_WEIGHTS = Path('experiments/5class_attention/checkpoints/best_model.weights.h5')
DIFF_WEIGHTS = Path('experiments/5class_diffattention/checkpoints/best_model.weights.h5')
LSTM_BASELINE_CSV = Path('experiments/5class_baseline/results/acc_per_snr.csv')
OUT_DIR = Path('experiments/5class_hybrid_confidence_intervals')

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())

In [ ]:
# CELL 2: Check required files and source weights
required = [
    'src/evaluate_hybrid_confidence_intervals.py',
    'src/models/mcldnn_attention.py',
    'src/models/mcldnn_diffattention.py',
    'configs/exp_5class_attention.yaml',
    'configs/exp_5class_diffattention.yaml',
    'src/train.py',
]

for f in required:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

print('Normal weights:', NORMAL_WEIGHTS, NORMAL_WEIGHTS.exists())
print('Diff weights  :', DIFF_WEIGHTS, DIFF_WEIGHTS.exists())
print('LSTM CSV      :', LSTM_BASELINE_CSV, LSTM_BASELINE_CSV.exists())

In [ ]:
# CELL 3: Train normal/diff models only if checkpoints are missing
# If your repo already contains checkpoints, this cell skips training.

def run_stream(cmd):
    print('Running:', ' '.join(map(str, cmd)))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, process.args)

jobs = [
    ('normal_attention', NORMAL_WEIGHTS, 'configs/exp_5class_attention.yaml'),
    ('diff_attention', DIFF_WEIGHTS, 'configs/exp_5class_diffattention.yaml'),
]

for name, weights, cfg in jobs:
    if weights.exists():
        print(f'{name}: checkpoint found, skipping training.')
    else:
        print(f'{name}: checkpoint missing, training now...')
        run_stream([sys.executable, '-u', 'src/train.py', '--config', cfg, '--datasetpath', str(DATASET)])
        assert weights.exists(), f'Missing after training: {weights}'

In [ ]:
# CELL 4: Run confidence interval analysis
if OUT_DIR.exists():
    print('Removing old CI outputs:', OUT_DIR)
    shutil.rmtree(OUT_DIR)

cmd = [
    sys.executable, '-u', 'src/evaluate_hybrid_confidence_intervals.py',
    '--datasetpath', str(DATASET),
    '--normal-weights', str(NORMAL_WEIGHTS),
    '--diff-weights', str(DIFF_WEIGHTS),
    '--lstm-baseline-csv', str(LSTM_BASELINE_CSV),
    '--output-dir', str(OUT_DIR),
    '--selection-mode', 'validation_best',
    '--low-snr-max', '2',
    '--bootstrap-samples', '5000',
]
run_stream(cmd)

assert (OUT_DIR / 'results/hybrid_confidence_intervals_per_snr.csv').exists()
assert (OUT_DIR / 'results/hybrid_confidence_interval_summary.json').exists()
print('Confidence interval analysis complete.')

In [ ]:
# CELL 5: Display CI tables
ci = pd.read_csv(OUT_DIR / 'results/hybrid_confidence_intervals_per_snr.csv')
summary = pd.read_json(OUT_DIR / 'results/hybrid_confidence_interval_summary.json', typ='series')

# Convert key columns to percent for easier reading
percent_cols = [c for c in ci.columns if 'accuracy' in c or 'ci_' in c or c.startswith('delta')]
ci_percent = ci.copy()
for c in percent_cols:
    if c in ci_percent.columns and pd.api.types.is_numeric_dtype(ci_percent[c]):
        ci_percent[c] = 100 * ci_percent[c]

print('Overall summary')
display(summary)

print('Per-SNR confidence intervals (%)')
display(ci_percent)

In [ ]:
# CELL 6: Display CI plots
figs = [
    OUT_DIR / 'figures/normal_vs_hybrid_accuracy_95ci_vs_snr.png',
    OUT_DIR / 'figures/lstm_normal_hybrid_accuracy_95ci_vs_snr.png',
    OUT_DIR / 'figures/hybrid_minus_normal_delta_95ci_by_snr.png',
]

for fig in figs:
    print(fig)
    display(Image(filename=str(fig)))

In [ ]:
# CELL 7: Create repo-ready zip for CI results only
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = Path('/kaggle/working') / f'hybrid_confidence_intervals_repo_ready_{stamp}'
zip_path = shutil.make_archive(
    str(zip_base),
    'zip',
    root_dir=str(WORK_DIR),
    base_dir='experiments/5class_hybrid_confidence_intervals',
)

print('Created repo-ready zip:', zip_path)
print('Extract this at repo root. It will create/update:')
print('  experiments/5class_hybrid_confidence_intervals/')
display(FileLink(zip_path))